<a href="https://colab.research.google.com/github/koushal-code/DecodeLabs-Project/blob/main/DecodeLabs_project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading the dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

In [ ]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d dhivyadharunaba/it-job-roles-skills-dataset --unzip
import pandas as pd
df = pd.read_csv("IT_Job_Roles_Skills.csv", encoding='latin-1')

cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/dhivyadharunaba/it-job-roles-skills-dataset
License(s): ODbL-1.0
100% 30.7k/30.7k [00:00<00:00, 45.8MB/s]



In [ ]:
df.head()

,Job Title,Job Description,Skills,Certifications
0,Admin Big Data,Responsible for managing and overseeing big da...,"Hadoop, Spark, MapReduce, Data Lakes, Data War...","Cloudera Certified Professional (CCP), Hortonw..."
1,Ansible Operations Engineer,Focuses on automating IT processes using Ansib...,"Ansible, Linux, Automation, Cloud Platforms, C...",Red Hat Certified Specialist in Ansible Automa...
2,Artifactory Administrator,Manages the Artifactory repository for build a...,"Artifactory, CI/CD, Jenkins, Docker, Maven, Gr...","JFrog Artifactory Certification, DevOps Instit..."
3,Artificial Intelligence / Machine Learning Leader,"Leads AI/ML projects and teams, defining strat...","AI Strategy, Machine Learning, Team Management...","AI-900: Microsoft Azure AI Fundamentals, Certi..."
4,Artificial Intelligence / Machine Learning Sr....,Senior role overseeing multiple AI/ML initiati...,"AI Strategy, Machine Learning, Team Management...",Certified Artificial Intelligence Practitioner...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Job Title        493 non-null    object
 1   Job Description  493 non-null    object
 2   Skills           493 non-null    object
 3   Certifications   493 non-null    object
dtypes: object(4)
memory usage: 15.5+ KB


In [ ]:
df.columns

Index(['Job Title', 'Job Description', 'Skills', 'Certifications'], dtype='object')

In [ ]:
df['Combined_Text'] = (
    df['Job Title'].fillna('') + " " +
    df['Job Description'].fillna('') + " " +
    df['Skills'].fillna('') + " " +
    df['Certifications'].fillna('')
)


def clean_text(text):
    text = text.lower()
    text = text.replace('ml', 'machine learning')
    text = text.replace('ai', 'artificial intelligence')
    text = text.replace('db', 'database')
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['Cleaned_Text'] = df['Combined_Text'].apply(clean_text)

#model fit
vectorizer = TfidfVectorizer(stop_words='english')
job_vectors = vectorizer.fit_transform(df['Cleaned_Text'])

In [ ]:
def get_user_skills():
    skill_suggestions = {
        "Languages": ["Python", "JavaScript", "Java", "C++", "Go", "Ruby"],
        "Frontend": ["React", "Angular", "Vue.js", "HTML", "CSS"],
        "Backend & DB": ["Node.js", "Django", "SQL", "MongoDB", "PostgreSQL"],
        "Cloud & DevOps": ["AWS", "Docker", "Kubernetes", "Linux", "CI/CD"],
        "Data & AI": ["Machine Learning", "Data Analysis", "TensorFlow", "Pandas"]
    }

    while True:
        print("\n" + "="*50)
        print("Enter your skills separated by commas (e.g., Python, SQL, AWS).")
        print("If you aren't sure what to type, just type 'help' to see suggestions.")

        user_input = input("Your Skills: ").strip()

        if user_input.lower() == 'help':
            print("\n--- POPULAR SKILLS TO CHOOSE FROM ---")
            for category, skills in skill_suggestions.items():
                print(f"** {category}: {', '.join(skills)}")
            print("-------------------------------------")
            continue

        elif len(user_input.replace(',', '').strip()) < 3:
            print("\n[!] Input too short. Please enter at least one valid skill.")
            continue

        else:
            return user_input

In [ ]:
print("Welcome to the Digital Career Matchmaker!")

# 1. Get the user input
user_input = get_user_skills()
print(f"\nAnalyzing roles matching: {user_input}...\n")

# 2. Clean and vectorize the input
cleaned_user_input = clean_text(user_input)
user_vector = vectorizer.transform([cleaned_user_input])

# 3. Calculate similarity scores
similarity_scores = cosine_similarity(user_vector, job_vectors).flatten()
df['Match_Score'] = similarity_scores

# 4. Standardize capitalization and drop duplicates
df['Job Title'] = df['Job Title'].str.title()
top_matches = df.sort_values(by='Match_Score', ascending=False)
deduplicated_matches = top_matches.drop_duplicates(subset=['Job Title'], keep='first')

# 5. Print the final recommendations
print("Top 3 Recommended Roles:")
print(deduplicated_matches[['Job Title', 'Match_Score']].head(3))

Welcome to the Digital Career Matchmaker!

Enter your skills separated by commas (e.g., Python, SQL, AWS).
If you aren't sure what to type, just type 'help' to see suggestions.
Your Skills: help

--- POPULAR SKILLS TO CHOOSE FROM ---
** Languages: Python, JavaScript, Java, C++, Go, Ruby
** Frontend: React, Angular, Vue.js, HTML, CSS
** Backend & DB: Node.js, Django, SQL, MongoDB, PostgreSQL
** Cloud & DevOps: AWS, Docker, Kubernetes, Linux, CI/CD
** Data & AI: Machine Learning, Data Analysis, TensorFlow, Pandas
-------------------------------------

Enter your skills separated by commas (e.g., Python, SQL, AWS).
If you aren't sure what to type, just type 'help' to see suggestions.
Your Skills: html, css, js

Analyzing roles matching: html, css, js...

Top 3 Recommended Roles:
                         Job Title  Match_Score
199  Web Designer (Ui/Ux Designer)     0.299534
124     Junior Front End Developer     0.299427
91       Entry Level Web Developer     0.263929
